In [ ]:
# Locate the repository when Jupyter starts in a notebook subdirectory.
from pathlib import Path
import sys

_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents)
              if (p / "figure" / "paths.py").is_file()
              and (p / "run_cross_validation.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the cloned sAge repository.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from figure.paths import input_path, output_path, font_path


# figure-2-5-human-heatmap-MLP

Plot human benchmark heatmaps with common color scales.

Run Jupyter from the repository root. Required external data and results are listed in `figure/INPUTS.md`. Set `SAGE_FIGURE_INPUT_ROOT` and `SAGE_FIGURE_OUTPUT_ROOT` when using other directories. See figure/VALIDATION.md for the execution checks and their limits.


Adapted from the mouse heatmap: keep panel dimensions and color styling aligned, and share a color scale for each metric across thresholds.

In [ ]:
# ============================================================
# FINAL HUMAN MLP HEATMAP CODE





# ============================================================

import os
import platform
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.font_manager as font_manager

# ==========================================

# ==========================================
system = platform.system()
if system == "Windows":
    font_path = sage_font_path()
elif system == "Darwin":
    font_path = sage_font_path()
else:
    font_path = sage_font_path()
    if not os.path.exists(font_path):
        font_path = sage_font_path()

if os.path.exists(font_path):
    font_manager.fontManager.addfont(font_path)
    plt.rcParams["font.family"] = "Arial"
else:
    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["axes.linewidth"] = 0.5

# ==========================================

# ==========================================
summary_csv_path = input_path("1-human-benchmark-model-output/summary_age_prediction/mlp_tanh/human_benchmark_mlp_tanh_prediction_summary.csv")
save_dir = output_path("1-human-benchmark-model-output/summary_age_prediction/mlp_tanh/2-heatmaps-human-from-mouse-code")
os.makedirs(save_dir, exist_ok=True)

models_to_include = {"scimmuaging", "buckley", "scale", "sage", "maple", "xgboost", "iage"}

# ==========================================

# ==========================================
colors_bwr = ["#1D4E89", "#73A5C6", "#FDFDFD", "#F26D5B", "#9E1A1A"]
cmap_bwr = mcolors.LinearSegmentedColormap.from_list("BlueWhiteRed", colors_bwr, N=256)
cmap_rwb = cmap_bwr.reversed(name="RedWhiteBlue")



MOUSE_COLOR_LIMITS = {
    "Precision": (0.0, 0.4, 0.05),
    "Pearson_R": (0.04411, 0.9661, 0.5525),
    "MAE": (1.054, 11.53, 5.668),
    "RMSE": (2.897, 11.9, 7.537),
}

metrics_config = {
    "Precision": {
        "cmap": cmap_bwr,
        "sort_ascending": False,
        "short_label": "Precision",
        "fmt": ".2f",
        "center_percentile": 60,
    },
    "Pearson_R": {
        "cmap": cmap_bwr,
        "sort_ascending": False,
        "short_label": "Pearson R",
        "fmt": ".2f",
        "center_percentile": 60,
    },
    "MAE": {
        "cmap": cmap_rwb,
        "sort_ascending": True,
        "short_label": "MAE(years)",
        "fmt": ".2f",
        "center_percentile": 40,
    },
    "RMSE": {
        "cmap": cmap_bwr,
        "sort_ascending": True,
        "short_label": "RMSE(years)",
        "fmt": ".2f",
        "center_percentile": 75,
    },
}

TISSUES_TO_EXCLUDE_BY_METRIC = {
    "Pearson_R": {"large intestine"},
    "MAE": {"large intestine"},
    "RMSE": {"large intestine"},
}


def clean_model_label(model):
    model_str = str(model).strip()
    model_lower = model_str.lower()
    if model_lower in {"sage", "maple"}:
        return "Sage"
    if model_lower == "scimmuaging":
        return "sc-ImmuAging"
    if model_lower == "iage":
        return "iAge"
    if model_lower == "xgboost":
        return "XGboost"
    return model_str[:1].upper() + model_str[1:].lower()


def clean_tissue_label(tissue):
    return str(tissue).replace("_", " ")


def clean_benchmark_label(benchmark_name):
    name = str(benchmark_name)
    lower = name.lower()
    if "genage" in lower:
        return "GenAge"
    if "cellage" in lower:
        return "CellAge"
    if "agingatlas" in lower:
        return "Aging Atlas"
    return name.replace(".xlsx", "").replace(".csv", "")


def safe_name(value):
    return str(value).replace("/", "-").replace(" ", "_").replace(".", "_")


def plot_one_heatmap(df_filtered, metric, config, threshold, benchmark_name=None, color_limits=None):
    value_df = df_filtered.dropna(subset=[metric]).copy()
    excluded_tissues = TISSUES_TO_EXCLUDE_BY_METRIC.get(metric, set())
    if excluded_tissues:
        tissue_labels = value_df["Tissue"].map(clean_tissue_label).str.lower()
        value_df = value_df[~tissue_labels.isin(excluded_tissues)].copy()
    if value_df.empty:
        return None

    pivot_df = value_df.pivot_table(index="Tissue", columns="Model", values=metric, aggfunc="mean")
    pivot_df = pivot_df.dropna(axis=0, how="all").dropna(axis=1, how="all")
    if pivot_df.empty:
        return None

    model_mean_scores = pivot_df.mean(axis=0).sort_values(ascending=config["sort_ascending"])
    pivot_df = pivot_df[model_mean_scores.index]


    if "sage" in pivot_df.columns:
        pivot_df = pivot_df.sort_values(by="sage", ascending=config["sort_ascending"])
        cols = list(pivot_df.columns)
        cols.insert(0, cols.pop(cols.index("sage")))
        pivot_df = pivot_df[cols]
    elif "maple" in pivot_df.columns:
        pivot_df = pivot_df.sort_values(by="maple", ascending=config["sort_ascending"])
        cols = list(pivot_df.columns)
        cols.insert(0, cols.pop(cols.index("maple")))
        pivot_df = pivot_df[cols]
    else:
        first_col = pivot_df.columns[0]
        pivot_df = pivot_df.sort_values(by=first_col, ascending=config["sort_ascending"])

    pivot_df.index = [clean_tissue_label(x) for x in pivot_df.index]
    pivot_df.columns = [clean_model_label(x) for x in pivot_df.columns]

    vals = pivot_df.values.astype(float)
    finite_vals = vals[np.isfinite(vals)]
    if finite_vals.size == 0:
        return None

    if color_limits is not None and metric in color_limits:
        global_min, global_max, plot_center = color_limits[metric]
    else:
        global_min = float(np.nanmin(finite_vals))
        global_max = float(np.nanmax(finite_vals))
        plot_center = float(np.nanpercentile(finite_vals, config["center_percentile"]))

    if not np.isfinite(global_min) or not np.isfinite(global_max) or global_max <= global_min:
        global_min = float(np.nanmin(finite_vals))
        global_max = float(np.nanmax(finite_vals))

    if not np.isfinite(global_min) or not np.isfinite(global_max) or global_max <= global_min:
        global_min, global_max = 0.0, 1.0

    if not np.isfinite(plot_center):
        plot_center = (global_min + global_max) / 2.0


    width_in = 100 / 25.4
    height_in = 80 / 25.4
    fig, ax = plt.subplots(figsize=(width_in, height_in))

    sns.heatmap(
        pivot_df,
        annot=True,
        cmap=config["cmap"],
        vmin=global_min,
        vmax=global_max,
        center=plot_center,
        robust=False,
        fmt=config["fmt"],
        annot_kws={"size": 6, "family": "Arial", "color": "black"},
        xticklabels=list(pivot_df.columns),
        yticklabels=list(pivot_df.index),
        cbar_kws={"shrink": 0.6, "aspect": 20, "pad": 0.04},
        ax=ax,
    )

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks(np.arange(pivot_df.shape[1]) + 0.5)
    ax.set_yticks(np.arange(pivot_df.shape[0]) + 0.5)
    ax.set_xticklabels(list(pivot_df.columns), rotation=45, ha="right", fontsize=6, family="Arial")
    ax.set_yticklabels(list(pivot_df.index), rotation=0, fontsize=6, family="Arial")
    ax.tick_params(axis="both", which="major", length=2.5, width=0.5, color="black", direction="out", pad=2)

    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=6, width=0.5, length=2.5, pad=2)
    cbar.outline.set_visible(False)
    cbar.ax.set_title(config["short_label"], size=6, family="Arial", pad=6, loc="left")

    if benchmark_name is not None:
        ax.set_title(clean_benchmark_label(benchmark_name), fontsize=7, pad=4)

    plt.tight_layout()

    if benchmark_name is None:
        save_name = f"Human_MLP_Tanh_Heatmap_{metric}_Threshold{threshold}.pdf"
    else:
        save_name = f"Human_MLP_Tanh_Heatmap_{metric}_Threshold{threshold}_{safe_name(clean_benchmark_label(benchmark_name))}.pdf"

    save_path = os.path.join(save_dir, save_name)
    plt.savefig(save_path, format="pdf", bbox_inches="tight", facecolor="white", transparent=False)
    plt.close(fig)
    return save_path

# ==========================================

# ==========================================
if not os.path.exists(summary_csv_path):
    print(f"找不到结果文件，请检查路径: {summary_csv_path}")
else:
    df_summary = pd.read_csv(summary_csv_path)
    df_summary = df_summary[
        df_summary["Model"].astype(str).str.lower().isin(models_to_include)
    ].copy()

    required_cols = {"Model", "Benchmark_Name", "Tissue", "Threshold", "Precision", "Pearson_R", "MAE", "RMSE"}
    missing_cols = required_cols.difference(df_summary.columns)
    if missing_cols:
        raise ValueError(f"summary CSV 缺少必要列: {sorted(missing_cols)}")

    all_thresholds = sorted(df_summary["Threshold"].dropna().unique())


    metric_color_limits = MOUSE_COLOR_LIMITS.copy()
    for metric, (vmin, vmax, center) in metric_color_limits.items():
        print(f"{metric} colorbar: vmin={vmin:.4g}, center={center:.4g}, vmax={vmax:.4g}")

    saved_files = []


    for threshold in all_thresholds:
        print(f"\n正在处理总体热图 Threshold = {threshold}")
        df_threshold = df_summary[df_summary["Threshold"] == threshold]
        if df_threshold.empty:
            continue
        for metric, config in metrics_config.items():
            save_path = plot_one_heatmap(
                df_threshold,
                metric,
                config,
                threshold,
                benchmark_name=None,
                color_limits=metric_color_limits,
            )
            if save_path:
                saved_files.append(save_path)
                print(f"    已保存: {save_path}")


    for threshold in all_thresholds:
        df_threshold = df_summary[df_summary["Threshold"] == threshold]
        for benchmark_name in sorted(df_threshold["Benchmark_Name"].dropna().unique()):
            df_bench = df_threshold[df_threshold["Benchmark_Name"] == benchmark_name]
            save_path = plot_one_heatmap(
                df_bench,
                "Precision",
                metrics_config["Precision"],
                threshold,
                benchmark_name=benchmark_name,
                color_limits=metric_color_limits,
            )
            if save_path:
                saved_files.append(save_path)
                print(f"    已保存 benchmark Precision: {save_path}")

    pd.DataFrame({"figure_path": saved_files}).to_csv(
        os.path.join(save_dir, "Human_MLP_Tanh_Heatmap_Figure_Index.csv"),
        index=False,
    )

    print(f"\n完成。共保存 {len(saved_files)} 张 PDF 热图。")
    print(f"输出目录: {save_dir}")
